# Notebook: nb_bronze_customers
# Purpose: Ingest customers into bronze lakehouse
# Layer: Bronze (raw ingestion)
# Engine: python (single-node polars / delta-rs)
# Source: csv - /lakehouse/default/Files/raw/customers/*.csv
# Target: bronze_customers (append-only)

In [ ]:
%run nb_utils_config

In [ ]:
# --- Imports ---
import os
import glob
from datetime import datetime, timezone

import polars as pl
from deltalake import write_deltalake
import notebookutils

In [ ]:
# --- Parameters ---
source_name = "customers"
source_format = "csv"  # csv | parquet | json
# File I/O uses the /lakehouse/default/... FUSE mount, NOT notebookutils.fs.ls
# on abfss:// (live-confirmed to hang ~90s then 500). See python-style-guide.md.
source_path = "/lakehouse/default/Files/raw/customers/*.csv"
load_mode = "append"  # bronze is append-only

In [ ]:
# --- Read Source Data (polars; discover files via the mount) ---
source_files = sorted(glob.glob(source_path))
assert source_files, f"No source files found at {source_path}"

df_raw = pl.concat(
    [pl.read_csv(f, infer_schema_length=10000) for f in source_files],
    how="diagonal_relaxed",
)

print(f"Source files: {len(source_files)}")
print(f"Source rows: {df_raw.height}")
print(f"Source columns: {df_raw.columns}")

In [ ]:
# --- Add Metadata Columns ---
# _load_timestamp : UTC load time literal (no per-row current_timestamp() UDF)
# _source_file    : resolved source path literal (no per-row input_file_name())
# _load_id        : Fabric run id via notebookutils.runtime.context
# `or "manual"`: currentRunId can be present-but-None interactively, and
# get(key, default) only returns the default when the key is ABSENT.
load_id = notebookutils.runtime.context.get("currentRunId") or "manual"
# dtype=pl.Utf8 is REQUIRED on the string lits: an untyped pl.lit(None) becomes a
# polars Null dtype -> Arrow null, which delta-rs rejects (SchemaMismatchError:
# Invalid data type for Delta Lake: Null).
df_bronze = df_raw.with_columns(
    pl.lit(datetime.now(timezone.utc)).alias("_load_timestamp"),
    pl.lit(source_path, dtype=pl.Utf8).alias("_source_file"),
    pl.lit(load_id, dtype=pl.Utf8).alias("_load_id"),
)

In [ ]:
# --- Write to Delta Table (append + schema merge; path via table_path()) ---
# **DELTA_WRITE_KWARGS (from nb_utils_config) selects the rust writer on delta-rs
# < 0.18, whose pyarrow writer rejects schema_mode; it is empty on newer delta-rs.
write_deltalake(
    table_path(f"bronze_{source_name}"),
    df_bronze.to_arrow(),
    mode="append",
    schema_mode="merge",
    **DELTA_WRITE_KWARGS,
)

print(f"Written to: bronze_{source_name}")

In [ ]:
# --- Validation ---
validate_row_count(f"bronze_{source_name}", min_rows=1)
print("PASS: Bronze load complete")